### DuckDB 연결

In [1]:
import duckdb
import pandas as pd
from pathlib import Path

DATA_PATH = Path('../data/raw')
DB_PATH = '../data/ecommerce_payinsight.duckdb'

con = duckdb.connect(DB_PATH)
print(f"DuckDB 연결 완료: {DB_PATH}")
print(f"DuckDB 버전: {duckdb.__version__}")

DuckDB 연결 완료: ../data/ecommerce_payinsight.duckdb
DuckDB 버전: 1.5.0


### 테이블 생성

## Star Schema 설계

### Dimension Tables

**dim_customer**

| 컬럼명 | 타입 | 설명 |
|---|---|---|
| customer_id | VARCHAR | 고객 ID (PK) |
| customer_unique_id | VARCHAR | 고객 고유 ID (동일인 중복 주문 구분) |
| customer_city | VARCHAR | 고객 거주 도시 |
| customer_state | VARCHAR | 고객 거주 주(state) 코드 |
| region_group | VARCHAR | 브라질 5개 권역 (파생) |

**dim_product**

| 컬럼명 | 타입 | 설명 |
|---|---|---|
| product_id | VARCHAR | 상품 ID (PK) |
| category_name_en | VARCHAR | 영문 카테고리명 |
| price_tier | VARCHAR | 가격 구간 (파생, Low/Mid/High/Premium) |
| weight_g | DOUBLE | 상품 무게 (g) |
| photos_qty | INTEGER | 상품 사진 수 |

**dim_seller**

| 컬럼명 | 타입 | 설명 |
|---|---|---|
| seller_id | VARCHAR | 판매자 ID (PK) |
| seller_city | VARCHAR | 판매자 위치 도시 |
| seller_state | VARCHAR | 판매자 위치 주(state) 코드 |
| region_group | VARCHAR | 브라질 5개 권역 (파생) |

**dim_date**

| 컬럼명 | 타입 | 설명 |
|---|---|---|
| date_id | INTEGER | 날짜 ID (PK, YYYYMMDD) |
| year | INTEGER | 연도 |
| month | INTEGER | 월 (1~12) |
| quarter | INTEGER | 분기 (1~4) |
| weekday | INTEGER | 요일 (0=월요일) |
| is_weekend | BOOLEAN | 주말 여부 |

**dim_campaign**

| 컬럼명 | 타입 | 설명 |
|---|---|---|
| campaign_id | INTEGER | 캠페인 ID (PK) |
| campaign_type | VARCHAR | 캠페인 유형 |
| description | VARCHAR | 캠페인 설명 |

---

### Fact Table

| 컬럼명 | 타입 | 설명 |
|---|---|---|
| order_id | VARCHAR | 주문 고유 ID (PK) |
| customer_id | VARCHAR | 고객 ID (FK → dim_customer) |
| product_id | VARCHAR | 상품 ID (FK → dim_product) |
| seller_id | VARCHAR | 판매자 ID (FK → dim_seller) |
| date_id | INTEGER | 날짜 ID (FK → dim_date, YYYYMMDD) |
| campaign_id | INTEGER | 캠페인 ID (FK → dim_campaign) |
| payment_type | VARCHAR | 결제수단 |
| payment_value | DOUBLE | 실결제금액 (BRL) |
| payment_installments | INTEGER | 할부 횟수 |
| freight_value | DOUBLE | 배송비 (BRL) |
| review_score | INTEGER | 고객평점 (1~5) |
| order_status | VARCHAR | 주문 상태 |
| delivery_days | INTEGER | 배송 소요 일수 (파생) |
| converted | INTEGER | 결제완료 여부 (파생, delivered=1) |
| is_delayed | BOOLEAN | 배송지연 여부 (파생, 14일 초과=True) |
| campaign_type | VARCHAR | 캠페인 유형 (파생) |

In [2]:
# dim_date
con.execute("""
CREATE TABLE IF NOT EXISTS dim_date (
    date_id      INTEGER PRIMARY KEY, -- YYYYMMDD 형식 (예: 20180101)
    year         INTEGER,             -- 연도
    month        INTEGER,             -- 월 (1~12)
    quarter      INTEGER,             -- 분기 (1~4)
    weekday      INTEGER,             -- 요일 (0=월 ~ 6=일)
    is_weekend   BOOLEAN              -- 주말 여부
)
""")

# dim_customer
con.execute("""
CREATE TABLE IF NOT EXISTS dim_customer (
    customer_id    VARCHAR PRIMARY KEY, -- 주문 단위 고객 ID
    customer_unique_id VARCHAR,         -- 실제 고객 고유 ID (재구매 추적용)
    customer_city  VARCHAR,             -- 고객 거주 도시
    customer_state VARCHAR,             -- 고객 거주 주(state) 코드
    region_group   VARCHAR              -- 브라질 5개 권역 (Southeast/Northeast/South/North/Central-West)
)
""")

# dim_product
con.execute("""
CREATE TABLE IF NOT EXISTS dim_product (
    product_id          VARCHAR PRIMARY KEY, -- 상품 고유 ID
    category_name_en    VARCHAR,             -- 영문 카테고리명 (category_translation 조인 결과)
    price_tier          VARCHAR,             -- 가격 구간 (Low/Mid/High/Premium)
    weight_g            FLOAT,               -- 상품 무게(g) - 배송비 영향
    photos_qty          INTEGER              -- 상품 사진 수 - 구매 전환율 영향
)
""")

# dim_seller
con.execute("""
CREATE TABLE IF NOT EXISTS dim_seller (
    seller_id     VARCHAR PRIMARY KEY, -- 판매자(가맹점) 고유 ID
    seller_city   VARCHAR,             -- 판매자 위치 도시
    seller_state  VARCHAR,             -- 판매자 위치 주(state) 코드
    region_group  VARCHAR              -- 판매자 권역
)
""")

# dim_campaign
con.execute("""
CREATE TABLE IF NOT EXISTS dim_campaign (
    campaign_id   INTEGER PRIMARY KEY, -- 캠페인 ID
    campaign_type VARCHAR,             -- 캠페인 유형 (Voucher_Campaign/Installment_Promo/Standard)
    description   VARCHAR              -- 캠페인 설명
)
""")

print("Dimension 테이블 5개 생성 완료")
con.execute("SHOW TABLES").df()

Dimension 테이블 5개 생성 완료


,name
0,dim_campaign
1,dim_customer
2,dim_date
3,dim_product
4,dim_seller


In [3]:
con.execute("""
CREATE TABLE IF NOT EXISTS fact_orders (
    order_id                VARCHAR PRIMARY KEY, -- 주문 고유 ID
    customer_id             VARCHAR,             -- 고객 ID (dim_customer 참조)
    product_id              VARCHAR,             -- 상품 ID (dim_product 참조)
    seller_id               VARCHAR,             -- 판매자 ID (dim_seller 참조)
    date_id                 INTEGER,             -- 날짜 ID (dim_date 참조, YYYYMMDD)
    campaign_id             INTEGER,             -- 캠페인 ID (dim_campaign 참조)
    payment_type            VARCHAR,             -- 결제 수단 (credit_card/boleto/voucher/debit_card)
    payment_value           FLOAT,               -- 실결제 금액 (BRL)
    payment_installments    INTEGER,             -- 할부 횟수 (1=일시불)
    freight_value           FLOAT,               -- 배송비 (BRL)
    review_score            INTEGER,             -- 고객 평점 (1~5)
    order_status            VARCHAR,             -- 주문 상태 (delivered/canceled 등)
    delivery_days           INTEGER,             -- 주문~배송 완료 소요 일수
    converted               INTEGER,             -- 결제 완료 여부 (delivered=1, 나머지=0)
    is_delayed              BOOLEAN,             -- 배송 지연 여부 (14일 초과=True)
    campaign_type           VARCHAR              -- 캠페인 유형 (dim_campaign 중복 저장, 조회 편의용)
)
""")

print("Fact 테이블 생성 완료")
con.execute("SHOW TABLES").df()

Fact 테이블 생성 완료


,name
0,dim_campaign
1,dim_customer
2,dim_date
3,dim_product
4,dim_seller
5,fact_orders


### 기초 데이터 생성

In [4]:
con.execute("""
INSERT INTO dim_campaign VALUES
    (1, 'Voucher_Campaign',   '바우처 결제 캠페인 - payment_type=voucher'),
    (2, 'Installment_Promo',  '할부 프로모션 - 6회 이상 할부 결제'),
    (3, 'Standard',           '일반 결제 - 프로모션 없음')
""")

result = con.execute("SELECT * FROM dim_campaign").df()
print(result)

   campaign_id      campaign_type                        description
0            1   Voucher_Campaign  바우처 결제 캠페인 - payment_type=voucher
1            2  Installment_Promo              할부 프로모션 - 6회 이상 할부 결제
2            3           Standard                    일반 결제 - 프로모션 없음


### 데이터 적재

In [5]:
import pandas as pd

# orders 날짜 범위 기반으로 dim_date 생성
orders = pd.read_csv('../data/raw/olist_orders_dataset.csv')
orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'])

date_range = pd.date_range(
    start=orders['order_purchase_timestamp'].min().date(),
    end=orders['order_purchase_timestamp'].max().date(),
    freq='D'
)

dim_date = pd.DataFrame({
    'date_id'   : [int(d.strftime('%Y%m%d')) for d in date_range],
    'year'      : [d.year for d in date_range],
    'month'     : [d.month for d in date_range],
    'quarter'   : [d.quarter for d in date_range],
    'weekday'   : [d.weekday() for d in date_range],
    'is_weekend': [d.weekday() >= 5 for d in date_range]
})

# 수정: 변수명을 직접 참조하는 방식으로 변경
con.execute("INSERT INTO dim_date SELECT * FROM dim_date")
print(f"dim_date 적재 완료: {len(dim_date):,}행")
print(f"날짜 범위: {dim_date['date_id'].min()} ~ {dim_date['date_id'].max()}")

dim_date 적재 완료: 774행
날짜 범위: 20160904 ~ 20181017


In [6]:
customers = pd.read_csv('../data/raw/olist_customers_dataset.csv')

region_map = {
    'SP': 'Southeast', 'RJ': 'Southeast', 'MG': 'Southeast', 'ES': 'Southeast',
    'BA': 'Northeast', 'CE': 'Northeast', 'PE': 'Northeast', 'MA': 'Northeast',
    'PB': 'Northeast', 'RN': 'Northeast', 'AL': 'Northeast', 'SE': 'Northeast',
    'PI': 'Northeast', 'RS': 'South', 'PR': 'South', 'SC': 'South',
    'PA': 'North', 'AM': 'North', 'RO': 'North', 'AC': 'North',
    'AP': 'North', 'RR': 'North', 'TO': 'North',
    'GO': 'Central-West', 'MT': 'Central-West', 'MS': 'Central-West', 'DF': 'Central-West'
}

customers['region_group'] = customers['customer_state'].map(region_map).fillna('Other')

dim_customer = customers[[
    'customer_id',
    'customer_unique_id',
    'customer_city',
    'customer_state',
    'region_group'
]].copy()

# register 방식으로 변경
con.execute("DELETE FROM dim_customer")
con.register('dim_customer_df', dim_customer)
con.execute("INSERT INTO dim_customer SELECT * FROM dim_customer_df")

print(f"dim_customer 적재 완료: {len(dim_customer):,}행")

result = con.execute("""
    SELECT region_group, COUNT(*) as cnt 
    FROM dim_customer 
    GROUP BY region_group 
    ORDER BY cnt DESC
""").df()
print(result)

dim_customer 적재 완료: 99,441행
   region_group    cnt
0     Southeast  68266
1         South  14148
2     Northeast   9394
3  Central-West   5782
4         North   1851


In [7]:
products = pd.read_csv('../data/raw/olist_products_dataset.csv')
category = pd.read_csv('../data/raw/product_category_name_translation.csv')

# 영문 카테고리명 조인
products = products.merge(category, on='product_category_name', how='left')
products['product_category_name_english'] = products['product_category_name_english'].fillna('Unknown')
products['product_weight_g'] = products['product_weight_g'].fillna(products['product_weight_g'].median())
products['product_photos_qty'] = products['product_photos_qty'].fillna(0)

# price_tier 생성
items = pd.read_csv('../data/raw/olist_order_items_dataset.csv')
avg_price = items.groupby('product_id')['price'].mean().reset_index()
avg_price.columns = ['product_id', 'avg_price']
products = products.merge(avg_price, on='product_id', how='left')

def get_price_tier(price):
    if pd.isna(price):  return 'Unknown'
    elif price < 50:    return 'Low'
    elif price < 150:   return 'Mid'
    elif price < 500:   return 'High'
    else:               return 'Premium'

products['price_tier'] = products['avg_price'].apply(get_price_tier)

dim_product = products[[
    'product_id',
    'product_category_name_english',
    'price_tier',
    'product_weight_g',
    'product_photos_qty'
]].copy()
dim_product.columns = ['product_id', 'category_name_en', 'price_tier', 'weight_g', 'photos_qty']

con.register('dim_product_df', dim_product)
con.execute("INSERT INTO dim_product SELECT * FROM dim_product_df")

print(f"dim_product 적재 완료: {len(dim_product):,}행")

result = con.execute("""
    SELECT price_tier, COUNT(*) as cnt 
    FROM dim_product 
    GROUP BY price_tier 
    ORDER BY cnt DESC
""").df()
print(result)

dim_product 적재 완료: 32,951행
  price_tier    cnt
0        Mid  13317
1        Low  11201
2       High   6910
3    Premium   1523


In [8]:
sellers = pd.read_csv('../data/raw/olist_sellers_dataset.csv')

region_map = {
    'SP': 'Southeast', 'RJ': 'Southeast', 'MG': 'Southeast', 'ES': 'Southeast',
    'BA': 'Northeast', 'CE': 'Northeast', 'PE': 'Northeast', 'MA': 'Northeast',
    'PB': 'Northeast', 'RN': 'Northeast', 'AL': 'Northeast', 'SE': 'Northeast',
    'PI': 'Northeast', 'RS': 'South', 'PR': 'South', 'SC': 'South',
    'PA': 'North', 'AM': 'North', 'RO': 'North', 'AC': 'North',
    'AP': 'North', 'RR': 'North', 'TO': 'North',
    'GO': 'Central-West', 'MT': 'Central-West', 'MS': 'Central-West', 'DF': 'Central-West'
}

sellers['region_group'] = sellers['seller_state'].map(region_map).fillna('Other')

dim_seller = sellers[[
    'seller_id',
    'seller_city',
    'seller_state',
    'region_group'
]].copy()

con.register('dim_seller_df', dim_seller)
con.execute("INSERT INTO dim_seller SELECT * FROM dim_seller_df")

print(f"dim_seller 적재 완료: {len(dim_seller):,}행")

result = con.execute("""
    SELECT region_group, COUNT(*) as cnt
    FROM dim_seller
    GROUP BY region_group
    ORDER BY cnt DESC
""").df()
print(result)

dim_seller 적재 완료: 3,095행
   region_group   cnt
0     Southeast  2287
1         South   668
2  Central-West    79
3     Northeast    56
4         North     5


In [9]:
orders   = pd.read_csv('../data/raw/olist_orders_dataset.csv')
payments = pd.read_csv('../data/raw/olist_order_payments_dataset.csv')
items    = pd.read_csv('../data/raw/olist_order_items_dataset.csv')
reviews  = pd.read_csv('../data/raw/olist_order_reviews_dataset.csv')

# 날짜 변환
orders['order_purchase_timestamp']      = pd.to_datetime(orders['order_purchase_timestamp'])
orders['order_delivered_customer_date'] = pd.to_datetime(orders['order_delivered_customer_date'])

# delivery_days 계산
orders['delivery_days'] = (
    orders['order_delivered_customer_date'] -
    orders['order_purchase_timestamp']
).dt.days

# 이상치 처리
orders.loc[orders['delivery_days'] > 100, 'delivery_days'] = None
orders.loc[orders['delivery_days'] == 0,  'delivery_days'] = 1

# converted, is_delayed
orders['converted']   = (orders['order_status'] == 'delivered').astype(int)
orders['is_delayed']  = orders['delivery_days'] > 14

# date_id
orders['date_id'] = orders['order_purchase_timestamp'].dt.strftime('%Y%m%d').astype(int)

# payments 집계 (order당 1행)
pay_agg = payments.groupby('order_id').agg(
    payment_type         = ('payment_type', 'first'),
    payment_value        = ('payment_value', 'sum'),
    payment_installments = ('payment_installments', 'max')
).reset_index()

# campaign_type 생성
def get_campaign(row):
    if row['payment_type'] == 'voucher':          return 'Voucher_Campaign'
    elif row['payment_installments'] >= 6:         return 'Installment_Promo'
    else:                                          return 'Standard'

pay_agg['campaign_type'] = pay_agg.apply(get_campaign, axis=1)
pay_agg['campaign_id']   = pay_agg['campaign_type'].map({
    'Voucher_Campaign': 1,
    'Installment_Promo': 2,
    'Standard': 3
})

# items 집계 (order당 1행 - 첫 번째 상품 기준)
items_agg = items.groupby('order_id').agg(
    product_id    = ('product_id', 'first'),
    seller_id     = ('seller_id', 'first'),
    freight_value = ('freight_value', 'sum')
).reset_index()

# reviews 집계 (order당 1행)
reviews_agg = reviews.groupby('order_id')['review_score'].mean().reset_index()

# 전체 조인
fact = orders[[
    'order_id', 'customer_id', 'date_id',
    'order_status', 'delivery_days', 'converted', 'is_delayed'
]].merge(pay_agg, on='order_id', how='left') \
  .merge(items_agg, on='order_id', how='left') \
  .merge(reviews_agg, on='order_id', how='left')

fact['review_score'] = fact['review_score'].round(0).astype('Int64')

fact_orders = fact[[
    'order_id', 'customer_id', 'product_id', 'seller_id', 'date_id',
    'campaign_id', 'payment_type', 'payment_value', 'payment_installments',
    'freight_value', 'review_score', 'order_status', 'delivery_days',
    'converted', 'is_delayed', 'campaign_type'
]]

con.register('fact_orders_df', fact_orders)
con.execute("INSERT INTO fact_orders SELECT * FROM fact_orders_df")

print(f"fact_orders 적재 완료: {len(fact_orders):,}행")

result = con.execute("""
    SELECT 
        COUNT(*)                                    as 총주문수,
        SUM(converted)                              as 결제완료,
        ROUND(AVG(delivery_days), 1)                as 평균배송일,
        ROUND(AVG(review_score), 2)                 as 평균리뷰점수,
        ROUND(AVG(payment_value), 0)                as 평균결제금액
    FROM fact_orders
""").df()
print(result)

fact_orders 적재 완료: 99,441행
    총주문수     결제완료  평균배송일  평균리뷰점수  평균결제금액
0  99441  96478.0   12.0    4.09   161.0


### Semantic Flat View

In [10]:
con.execute("""
CREATE OR REPLACE VIEW vw_order_full AS
SELECT
    f.order_id,
    f.order_status,
    f.delivery_days,          -- 주문~배송완료 소요 일수
    f.converted,              -- 결제완료=1, 나머지=0
    f.is_delayed,             -- 배송지연 여부 (14일 초과=True)
    f.payment_type,           -- 결제수단 (credit_card/boleto/voucher/debit_card)
    f.payment_value,          -- 실결제금액 (BRL)
    f.payment_installments,   -- 할부횟수 (1=일시불)
    f.freight_value,          -- 배송비 (BRL)
    f.review_score,           -- 고객평점 (1~5)
    f.campaign_type,          -- 캠페인유형 (Voucher_Campaign/Installment_Promo/Standard)
    c.customer_city,
    c.customer_state,
    c.region_group,           -- 브라질 5개 권역
    p.category_name_en,       -- 영문 카테고리명
    p.price_tier,             -- 가격구간 (Low/Mid/High/Premium)
    s.seller_city,
    s.seller_state,
    d.year,
    d.month,
    d.quarter,
    d.is_weekend
FROM fact_orders f
LEFT JOIN dim_customer c ON f.customer_id = c.customer_id
LEFT JOIN dim_product  p ON f.product_id  = p.product_id
LEFT JOIN dim_seller   s ON f.seller_id   = s.seller_id
LEFT JOIN dim_date     d ON f.date_id     = d.date_id
""")

print("vw_order_full 생성 완료")

# 테스트 쿼리 3개
print("\n[ 테스트 1 ] 결제수단별 평균 결제금액")
print(con.execute("""
    SELECT payment_type, COUNT(*) as cnt, ROUND(AVG(payment_value), 0) as avg_value
    FROM vw_order_full
    GROUP BY payment_type
    ORDER BY cnt DESC
""").df())

print("\n[ 테스트 2 ] 지역별 평균 배송일")
print(con.execute("""
    SELECT region_group, ROUND(AVG(delivery_days), 1) as avg_days
    FROM vw_order_full
    WHERE delivery_days IS NOT NULL
    GROUP BY region_group
    ORDER BY avg_days DESC
""").df())

print("\n[ 테스트 3 ] 캠페인별 전환율")
print(con.execute("""
    SELECT campaign_type, ROUND(AVG(converted)*100, 1) as cvr_pct
    FROM vw_order_full
    GROUP BY campaign_type
    ORDER BY cvr_pct DESC
""").df())

vw_order_full 생성 완료

[ 테스트 1 ] 결제수단별 평균 결제금액
  payment_type    cnt  avg_value
0  credit_card  75387      167.0
1       boleto  19784      145.0
2      voucher   2739      131.0
3   debit_card   1527      143.0
4  not_defined      3        0.0
5         None      1        NaN

[ 테스트 2 ] 지역별 평균 배송일
   region_group  avg_days
0         North      21.8
1     Northeast      19.2
2  Central-West      14.5
3         South      13.5
4     Southeast      10.2

[ 테스트 3 ] 캠페인별 전환율
       campaign_type  cvr_pct
0               None    100.0
1           Standard     97.2
2  Installment_Promo     96.7
3   Voucher_Campaign     94.3


### 메타데이터 생성

In [11]:
import json

metadata = {
    "tables": {
        "vw_order_full": {
            "description": "주문/결제/고객/상품/판매자를 통합한 Semantic Flat View. Text2SQL 질의의 기본 대상 테이블.",
            "columns": {
                "order_id":               "주문 고유 ID",
                "order_status":           "주문 상태 (delivered/canceled/shipped 등)",
                "delivery_days":          "주문~배송완료 소요 일수. NULL이면 미배송",
                "converted":              "결제완료 여부. delivered=1, 나머지=0",
                "is_delayed":             "배송지연 여부. 14일 초과=True",
                "payment_type":           "결제수단. credit_card/boleto/voucher/debit_card",
                "payment_value":          "실결제금액 (BRL 기준)",
                "payment_installments":   "할부횟수. 1=일시불, 6 이상=장기할부",
                "freight_value":          "배송비 (BRL 기준)",
                "review_score":           "고객평점. 1~5점, 5점이 최고",
                "campaign_type":          "캠페인유형. Voucher_Campaign/Installment_Promo/Standard",
                "customer_city":          "고객 거주 도시",
                "customer_state":         "고객 거주 주(state) 코드. 예: SP=상파울루",
                "region_group":           "브라질 5개 권역. Southeast/Northeast/South/North/Central-West",
                "category_name_en":       "영문 상품 카테고리명. 예: electronics, furniture",
                "price_tier":             "가격구간. Low(<50)/Mid(<150)/High(<500)/Premium(500+)",
                "seller_city":            "판매자(가맹점) 위치 도시",
                "seller_state":           "판매자(가맹점) 위치 주(state) 코드",
                "year":                   "주문 연도",
                "month":                  "주문 월 (1~12)",
                "quarter":                "주문 분기 (1~4)",
                "is_weekend":             "주말 주문 여부. True=주말"
            }
        },
        "fact_orders": {
            "description": "주문 팩트 테이블. vw_order_full로 조회 불가한 경우에만 사용.",
            "columns": {
                "order_id":             "주문 고유 ID",
                "customer_id":          "고객 ID (dim_customer 참조)",
                "product_id":           "상품 ID (dim_product 참조)",
                "seller_id":            "판매자 ID (dim_seller 참조)",
                "date_id":              "날짜 ID (dim_date 참조, YYYYMMDD)",
                "campaign_id":          "캠페인 ID (dim_campaign 참조)",
                "payment_type":         "결제수단",
                "payment_value":        "실결제금액 (BRL)",
                "payment_installments": "할부횟수",
                "freight_value":        "배송비 (BRL)",
                "review_score":         "고객평점 1~5",
                "order_status":         "주문상태",
                "delivery_days":        "배송 소요 일수",
                "converted":            "결제완료=1, 나머지=0",
                "is_delayed":           "배송지연=True",
                "campaign_type":        "캠페인유형"
            }
        }
    },
    "sample_questions": [
        {
            "question": "결제수단별 평균 결제금액은?",
            "sql": "SELECT payment_type, ROUND(AVG(payment_value), 0) as avg_value FROM vw_order_full GROUP BY payment_type ORDER BY avg_value DESC"
        },
        {
            "question": "지역별 평균 배송일은?",
            "sql": "SELECT region_group, ROUND(AVG(delivery_days), 1) as avg_days FROM vw_order_full WHERE delivery_days IS NOT NULL GROUP BY region_group ORDER BY avg_days DESC"
        },
        {
            "question": "배송 지연된 주문의 평균 리뷰 점수는?",
            "sql": "SELECT is_delayed, ROUND(AVG(review_score), 2) as avg_score FROM vw_order_full WHERE review_score IS NOT NULL GROUP BY is_delayed"
        },
        {
            "question": "캠페인별 전환율은?",
            "sql": "SELECT campaign_type, ROUND(AVG(converted)*100, 1) as cvr_pct FROM vw_order_full GROUP BY campaign_type ORDER BY cvr_pct DESC"
        },
        {
            "question": "Southeast 지역에서 가장 많이 팔린 카테고리 Top 5는?",
            "sql": "SELECT category_name_en, COUNT(*) as cnt FROM vw_order_full WHERE region_group = 'Southeast' GROUP BY category_name_en ORDER BY cnt DESC LIMIT 5"
        }
    ]
}

# JSON 파일 저장
with open('../docs/schema_metadata.json', 'w', encoding='utf-8') as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print("schema_metadata.json 저장 완료")
print(f"테이블 수: {len(metadata['tables'])}개")
print(f"샘플 질문 수: {len(metadata['sample_questions'])}개")

schema_metadata.json 저장 완료
테이블 수: 2개
샘플 질문 수: 5개
